# Image Stitching - Refactored

This notebook demonstrates image stitching using Harris corner detection, SIFT features, and homography.

## Features
- Harris corner detection for feature point identification
- SIFT (Scale-Invariant Feature Transform) for feature description
- Feature matching with ratio test
- Homography computation with RANSAC
- Multi-band blending for seamless panoramas

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from image_stitching import ImageStitcher

## Configure Image Paths

Define paths to image pairs for stitching.

In [ ]:
# Base directory for images
IMAGE_DIR = "image pairs"

# Define image pairs
image_pairs = [
    (os.path.join(IMAGE_DIR, "image pairs_01_01.jpg"),
     os.path.join(IMAGE_DIR, "image pairs_01_02.jpg")),
    (os.path.join(IMAGE_DIR, "image pairs_02_01.png"),
     os.path.join(IMAGE_DIR, "image pairs_02_02.png")),
    (os.path.join(IMAGE_DIR, "image pairs_03_01.jpg"),
     os.path.join(IMAGE_DIR, "image pairs_03_02.jpg")),
    (os.path.join(IMAGE_DIR, "image pairs_04_01.jpg"),
     os.path.join(IMAGE_DIR, "image pairs_04_02.jpg")),
]

## Visualize Original Images

Display all image pairs before stitching.

In [ ]:
def display_image_pairs(image_pairs):
    """Display all image pairs in a grid."""
    num_pairs = len(image_pairs)
    fig, axes = plt.subplots(num_pairs, 2, figsize=(10, 3 * num_pairs))
    
    if num_pairs == 1:
        axes = axes.reshape(1, -1)
    
    for i, (img1_path, img2_path) in enumerate(image_pairs):
        img1 = plt.imread(img1_path)
        img2 = plt.imread(img2_path)
        
        axes[i, 0].set_title(f'Pair {i+1} - Image A')
        axes[i, 0].imshow(img1)
        axes[i, 0].axis('off')
        
        axes[i, 1].set_title(f'Pair {i+1} - Image B')
        axes[i, 1].imshow(img2)
        axes[i, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

display_image_pairs(image_pairs)

## Harris Corner Detection

Visualize corners detected using the Harris corner detector.

In [ ]:
def display_corner_detection(image_pairs, stitcher):
    """Display corner detection results for all images."""
    num_pairs = len(image_pairs)
    fig, axes = plt.subplots(num_pairs, 2, figsize=(10, 3 * num_pairs))
    
    if num_pairs == 1:
        axes = axes.reshape(1, -1)
    
    for i, (img1_path, img2_path) in enumerate(image_pairs):
        # Detect corners
        img1_corners, corners1 = stitcher.detect_corners(img1_path)
        img2_corners, corners2 = stitcher.detect_corners(img2_path)
        
        axes[i, 0].set_title(f'Pair {i+1}A - {len(corners1)} corners')
        axes[i, 0].imshow(img1_corners)
        axes[i, 0].axis('off')
        
        axes[i, 1].set_title(f'Pair {i+1}B - {len(corners2)} corners')
        axes[i, 1].imshow(img2_corners)
        axes[i, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Initialize stitcher
stitcher = ImageStitcher()

# Display corner detection results
display_corner_detection(image_pairs, stitcher)

## Image Stitching

Stitch image pairs to create panoramic views.

In [ ]:
def stitch_and_display(image_pairs, stitcher):
    """Stitch all image pairs and display results."""
    results = []
    
    for i, (img1_path, img2_path) in enumerate(image_pairs):
        print(f"Stitching pair {i+1}...")
        try:
            result = stitcher.stitch_images(img1_path, img2_path)
            results.append(result)
            print(f"  Success! Result shape: {result.shape}")
        except Exception as e:
            print(f"  Error: {e}")
            results.append(None)
    
    # Display results
    fig, axes = plt.subplots(len(results), 1, figsize=(12, 4 * len(results)))
    
    if len(results) == 1:
        axes = [axes]
    
    for i, result in enumerate(results):
        if result is not None:
            axes[i].set_title(f'Stitched Panorama - Pair {i+1}')
            axes[i].imshow(result)
        else:
            axes[i].set_title(f'Stitching Failed - Pair {i+1}')
            axes[i].text(0.5, 0.5, 'Failed to stitch', 
                        ha='center', va='center')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Perform stitching
results = stitch_and_display(image_pairs, stitcher)

## Summary

This notebook demonstrates a complete image stitching pipeline:

1. **Corner Detection**: Harris corner detector identifies feature points
2. **Feature Description**: SIFT extracts robust descriptors at corner locations
3. **Feature Matching**: Brute-force matcher with ratio test finds correspondences
4. **Homography Estimation**: RANSAC computes the transformation matrix
5. **Image Blending**: Gradient masks create seamless panoramas

The refactored code is now modular, well-documented, and easy to maintain.